# Show4DSTEM

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/docs/tutorials/show4dstem.ipynb)

`Show4DSTEM` opens a 4D-STEM dataset with live virtual detectors: a bright-field / annular-dark-field aperture over the diffraction stack on one side, the resulting virtual image on the other. It accepts a NumPy array, a PyTorch tensor, a quantem `Dataset4dstem`, or the output of `load(...)`.

This tutorial uses the public binned gold 4D-STEM dataset from [`bobleesj/quantem-data`](https://huggingface.co/datasets/bobleesj/quantem-data). The built-in tutorial loader returns a calibrated real-data preview by striding the `gold_128_npy_bin8` scan to 64 by 64 positions with 24 by 24 detector frames, so the rendered documentation opens quickly while preserving uint16 detector counts. Use `load_tutorial_show4dstem(scan_stride=1)` for the full 128 by 128 scan.

```{tip}
Run this exact notebook with the Colab badge above, or [View or download this notebook on GitHub](https://github.com/bobleesj/quantem.widget/blob/main/docs/tutorials/show4dstem.ipynb). For finished results, use [Saving and sharing](widget_export) to export interactive HTML or share a trusted notebook with widget state.
```


In [ ]:
import subprocess
import sys

import numpy as np

try:
    import google.colab  # noqa: F401
except Exception:
    pass
else:
    from google.colab import output

    output.enable_custom_widget_manager()
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/bobleesj/quantem.widget.git"],
        check=True,
    )

from quantem.widget import Show4DSTEM
from quantem.widget.data import load_tutorial_show4dstem

stem_dataset = load_tutorial_show4dstem(scan_stride=2)


## Virtual detectors on real gold data

Drag the bright-field disk across the diffraction pattern, or use the BF / ABF / ADF presets, and watch the virtual image update from the real binned 4D-STEM stack. The documentation view embeds the compact browser data payload so the widget remains interactive after the page is rendered.

In [ ]:
Show4DSTEM(
    stem_dataset,
    offline=True,
    offline_dtype="uint16",
    save_state=True,
    precompute_virtual_images=False,
    show_fft=False,
)

## Multi-panel screening

Use `view_mode="multiple"` when the extra frame axis represents datasets, time points, scan regions, or acquisition repeats that should share one detector ROI. The diffraction panel stays the familiar Show4DSTEM control surface; the virtual-image side becomes a grid. Click a tile to make it the selected dataset, switch `compare_dp_mode` between `"average"` and `"selected"`, and use the tile star/hide/reorder controls to curate the set for a later cell or export.

The small example below splits the same real gold scan into four calibrated regions so the tutorial remains light, but the API is the same for a list of real master files.


In [ ]:
gold = stem_dataset.array
mid_row = gold.shape[0] // 2
mid_col = gold.shape[1] // 2

region_stack = np.stack(
    [
        gold[:mid_row, :mid_col],
        gold[:mid_row, mid_col:],
        gold[mid_row:, :mid_col],
        gold[mid_row:, mid_col:],
    ],
    axis=0,
).astype(gold.dtype, copy=False)

region_labels = ["upper-left", "upper-right", "lower-left", "lower-right"]

multi = Show4DSTEM(
    region_stack,
    sampling=tuple(stem_dataset.sampling),
    units=tuple(stem_dataset.units),
    view_mode="multiple",
    compare_cols=2,
    compare_max_panels=4,
    compare_panel_gap_px=0,
    compare_dp_mode="average",
    frame_dim_label="Region",
    frame_labels=region_labels,
    offline=True,
    offline_dtype="uint16",
    save_state=True,
    precompute_virtual_images=False,
    show_fft=False,
)
multi


## Real master-file folders on a workstation

For large no-bin or multi-acquisition sessions, load completed `*_master.h5` files directly and keep the comparison grid dense. Pass `sampling` and `units` when the file metadata does not contain physical scan or reciprocal-space calibration. Use pixel units when calibration is unknown rather than inventing mrad or angstrom values.

```python
from pathlib import Path
from quantem.widget import Show4DSTEM, load

folder = Path("/data/session/maped")
masters = sorted(str(path) for path in folder.glob("*_master.h5"))

# Replace these with measured calibration when available.
sampling = (1.0, 1.0, 1.0, 1.0)
units = ("pixels", "pixels", "pixels", "pixels")

data = load(
    masters,
    backend="cuda",
    devices=[0, 1],
    det_bin=1,
    series_type="generic",
    sampling=sampling,
    units=units,
)

viewer = Show4DSTEM(
    data,
    view_mode="multiple",
    compare_cols=3,
    compare_max_panels=len(masters),
    compare_panel_gap_px=0,
    compare_dp_mode="average",
    sampling=sampling,
    units=units,
)
viewer
```

Use `compare_dp_mode="average"` to show the average diffraction pattern for the visible grid, or `compare_dp_mode="selected"` when clicking a panel should show that dataset's diffraction pattern. The panel order, hidden panels, and starred panels are stored on the widget and can be reused with `state_dict()` / `load_state_dict()`.
